# IBR-FaultScreen in your browser

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ishaibu/ibr-faultscreen/blob/main/examples/colab_quickstart.ipynb)

Run real fault simulations on the IEEE 39-bus system with inverter-based resources (IBRs),
with nothing to install on your computer. This notebook runs on Google's free servers.

**How to use it:** run the cells from top to bottom. Click the ▶ button at the left of each
cell, or use **Runtime → Run all**. Each step says roughly how long it takes (free Colab
machines can be slower).

> **Screening only.** Generic WECC IBR models and illustrative limits on a test system.
> The results prioritize detailed studies; they do not replace an interconnection study.

No time to run anything? The [interactive demo](https://ishaibu.github.io/ibr-faultscreen/demo/) has every combination of IBRs, fault
bus and clearing time already simulated ("Build your own grid"), and a quiz.

Project: [github.com/ishaibu/ibr-faultscreen](https://github.com/ishaibu/ibr-faultscreen) ·
Cite: [doi.org/10.5281/zenodo.23290219](https://doi.org/10.5281/zenodo.23290219)

## 1. Install IBR-FaultScreen (about 1 minute)

This installs the tool and its simulator (ANDES) from PyPI. The output ends with the
installed versions.

If Colab shows a message asking you to **restart the session**, click it, wait a few
seconds, then continue with step 2 (no need to repeat step 1).

In [ ]:
%pip install -q "ibr-faultscreen>=0.2.0"
from importlib.metadata import version
print("IBR-FaultScreen", version("ibr-faultscreen"), "· ANDES", version("andes"))

## 2. Run the five example scenarios (2 to 4 minutes)

Five three-phase faults on the IEEE 39-bus system, with zero to three synchronous machines
replaced by IBRs. Each line shows the verdict (RISKY or OK) and how the simulation ended.
The first run is slower while ANDES prepares its models.

In [ ]:
!ibr-faultscreen example --no-open --out example_output

## 3. Look at the results

One row per scenario: which generators became IBRs, the fault, the verdict and why, and the
lowest short-circuit ratio (SCR) among the IBRs, one of the pre-fault features the screening
model uses (99 means the scenario has no IBRs).

**"RISKY (unresolved)"** means the simulator could not solve this bolted (zero-impedance)
fault, and the milder-fault check could not decide either way. The tool reports this
honestly instead of guessing: such scenarios need a detailed study.

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 120)
results = pd.read_csv("example_output/results.csv").round({"scr_min": 2})
results[["scenario_id", "ibrs", "fault_bus", "clear_time_s", "label", "resolution",
         "status", "scr_min", "reasons"]].fillna("")

## 4. Open the interactive report

The same report the tool makes on a computer: summary, network map, ranked table, and
trajectories you can replay, including the animated equal-area view. It opens below; you can
also download it (next cell) and open it in any browser.

In [ ]:
from IPython.display import HTML, display
display(HTML(open("example_output/report.html", encoding="utf-8").read()))

In [ ]:
# Download the report and the results table to your computer (works in Google Colab).
try:
    from google.colab import files
    files.download("example_output/report.html")
    files.download("example_output/results.csv")
except ImportError:
    print("Not running in Colab: the files are in the example_output folder.")

## 5. Try your own scenario (about 10 to 30 seconds per run)

Move the sliders on the right (or edit the code), then run the cell.

- **fault_bus:** a network bus, 1 to 29.
- **clearing_cycles:** fault duration in cycles at 60 Hz (6 cycles = 100 ms).
- **ibr_generators:** generators to replace with IBRs, from 1 to 9, separated by commas.
  Leave it empty for no IBRs. Generator 10 is the slack machine and cannot be replaced.
- **ibr_kind:** `solar` or `bess` (battery storage, discharging; same generic models).
- **load_scale:** system load as a multiple of the base case (0.9 = light, 1.1 = heavy).

The figure has five panels: bus voltages, rotor-angle spread, frequency, IBR current, and the
**equal-area view** (see step 6).

In [ ]:
fault_bus = 16  # @param {type:"slider", min:1, max:29, step:1}
clearing_cycles = 6  # @param {type:"slider", min:3, max:18, step:1}
ibr_generators = "2, 6"  # @param {type:"string"}
ibr_kind = "solar"  # @param ["solar", "bess"]
load_scale = 1.0  # @param {type:"slider", min:0.9, max:1.1, step:0.01}

import logging
import andes
import pandas as pd
from IPython.display import Image, display
from ibr_faultscreen.criteria import default_criteria
from ibr_faultscreen.equal_area import equal_area
from ibr_faultscreen.label import label
from ibr_faultscreen.plots import plot_scenario
from ibr_faultscreen.scenario import FaultSpec, IBRSpec, Scenario
from ibr_faultscreen.simulate import simulate

andes.config_logger(stream_level=50)
logging.disable(logging.WARNING)
settings = default_criteria()


def parse_gens(text):
    return sorted({int(g) for g in text.replace(" ", "").split(",") if g})


def make_scenario(bus, cycles, gens, kind="solar", load=1.0, name=None):
    name = name or f"b{bus}_c{cycles}_" + ("ibr" + "".join(map(str, gens)) if gens else "no_ibr")
    return Scenario(name, load_scale=load, ibrs=tuple(IBRSpec(g, kind=kind) for g in gens),
                    fault=FaultSpec(bus, clear_time=cycles / 60))


gens = parse_gens(ibr_generators)
scenario = make_scenario(fault_bus, clearing_cycles, gens, ibr_kind, load_scale)
result = simulate(scenario)
verdict = label(result, settings)
print("Verdict:", "RISKY" if verdict.risky else "OK",
      f"({verdict.resolution})" if verdict.resolution else "")
print("Simulation status:", result.status)
for reason in verdict.reasons:
    print(" -", reason)
display(Image(filename=str(plot_scenario(result, verdict, settings, f"{scenario.scenario_id}.png"))))

**Reading the figure:** dashed lines are the screening limits. A "RISKY (unresolved)" verdict
means the solver could not settle the bolted fault and the milder-fault check could not
decide; the tool says so instead of guessing.

## 6. The equal-area view, in words

The fifth panel reduces the synchronous machines to one equivalent machine (the group that
swings ahead against the rest) and draws the energy it gains while the fault is on
(accelerating area) and gives back after clearing (decelerating area). If the swing stops,
the two areas are equal; if it cannot give the energy back, the machines lose step. IBRs have
no rotor, so they are not machines here: they change the picture through the network, by
removing the inertia of the machines they replace and by the current they inject.

In [ ]:
print(equal_area(result if result.status != "sim_fail" else (result.solved_diagnostic() or result)).text())

## 7. Compare two scenarios (about 30 seconds)

Same fault, two different sets of IBRs (or any other change). The table shows the verdicts and
the equal-area numbers side by side. With the defaults, the fault at bus 16 cleared in 6 cycles
is OK with no IBRs and RISKY with IBRs on generators 8 and 9 (overvoltage at an IBR bus after
clearing), while the rotor angles stay in step in both.

In [ ]:
ibrs_a = ""  # @param {type:"string"}
ibrs_b = "8, 9"  # @param {type:"string"}
compare_bus = 16  # @param {type:"slider", min:1, max:29, step:1}
compare_cycles = 6  # @param {type:"slider", min:3, max:18, step:1}

import pandas as pd
from IPython.display import Image, display

rows = []
for tag, text in (("A", ibrs_a), ("B", ibrs_b)):
    sc = make_scenario(compare_bus, compare_cycles, parse_gens(text), name=f"compare_{tag}")
    r = simulate(sc)
    lab = label(r, settings)
    shown = r if r.status != "sim_fail" else (r.solved_diagnostic() or r)
    ea = equal_area(shown)
    rows.append({"case": tag, "IBRs on generators": text or "none",
                 "verdict": ("RISKY" if lab.risky else "OK") + (f" ({lab.resolution})" if lab.resolution else ""),
                 "status": r.status,
                 "accelerating area (pu·rad)": round(ea.accelerating, 2) if ea.applicable else None,
                 "decelerating area (pu·rad)": round(ea.decelerating, 2) if ea.applicable else None,
                 "swing stops at (°)": round(ea.delta_end_deg) if ea.applicable and ea.stable else None,
                 "first swing": ("stable" if ea.stable else "unstable") if ea.applicable else ea.reason})
    plot_scenario(r, lab, settings, f"compare_{tag}.png")
display(pd.DataFrame(rows).set_index("case"))
for tag in ("A", "B"):
    print(f"Case {tag}:")
    display(Image(filename=f"compare_{tag}.png", width=650))

## 8. Risk versus clearing time (1 to 3 minutes)

Runs the scenario from step 5 at several clearing times and charts the largest rotor-angle
spread; each point is colored by its verdict.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, display

cycles_list = [3, 5, 6, 9, 12, 15, 18]
points = []
for c in cycles_list:
    r = simulate(make_scenario(fault_bus, c, gens, ibr_kind, load_scale))
    lab = label(r, settings)
    shown = r if r.status != "sim_fail" else r.solved_diagnostic()   # milder fault if bolted failed
    tr = None if shown is None else shown.traj
    spread = (float(np.degrees(tr.delta.max(1) - tr.delta.min(1)).max())
              if tr is not None and tr.delta.shape[1] >= 2 else None)
    points.append((c, spread, lab.risky, r.status == "sim_fail"))
    print(f"{c:2d} cycles: {'RISKY' if lab.risky else 'OK':5s} {lab.resolution or ''} [{r.status}]")

fig, ax = plt.subplots(figsize=(7.5, 3.8))
xs = [c for c, s, *_ in points if s is not None]
ax.plot(xs, [s for c, s, *_ in points if s is not None], color="#9a9993", linewidth=1, zorder=2)
for c, s, risky, milder in points:
    color = "#b8401c" if risky else "#1b7a46"
    if s is None:
        ax.text(c, 0.04, "unsolvable", transform=ax.get_xaxis_transform(), ha="center", fontsize=8, color="#55544f")
        continue
    ax.scatter(c, s, s=70, zorder=3, facecolor="white" if milder else color, edgecolor=color, linewidth=2)
ax.axhline(180, color="#0b0b0b", linestyle="--", linewidth=1)
ax.text(cycles_list[-1] + 0.4, 172, "angle limit 180°", fontsize=8, ha="right")
ax.set_xlim(cycles_list[0] - 1, cycles_list[-1] + 1)
ax.set_xticks(cycles_list)
ax.set_ylim(0, max(200, max([s for s in (p[1] for p in points) if s is not None] + [0]) * 1.1))
ax.set_xlabel("Clearing time (cycles at 60 Hz)")
ax.set_ylabel("Largest rotor-angle spread (°)")
ax.set_title(f"Fault at bus {fault_bus}, " + (f"IBRs on generators {', '.join(map(str, gens))}" if gens else "no IBRs")
             + "\ngreen = OK, red = RISKY; hollow = bolted fault unsolvable, milder fault shown",
             fontsize=9.5, loc="left")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig("clearing_time_sweep.png", dpi=120)   # saved and shown as an image: works in any notebook
plt.close(fig)
display(Image(filename="clearing_time_sweep.png"))

## 9. Critical clearing time at one bus (about 1 minute; 3 to 5 minutes with IBRs)

The critical clearing time (CCT) is the longest fault duration after which the synchronous
machines stay in step, found by bisection. The command also draws the equal-area view just
below and just above the CCT. Leave **cct_ibr_generators** empty for the quick version.

If the bolted fault cannot be solved beyond some duration, the answer is given as "at
least …", with the CCT of a slightly milder fault as an upper estimate.

In [ ]:
cct_bus = 16  # @param {type:"slider", min:1, max:29, step:1}
cct_ibr_generators = ""  # @param {type:"string"}

import subprocess, os
args = ["ibr-faultscreen", "cct", "--bus", str(cct_bus), "--t-max", "0.3", "--out", "cct_output/cct.csv"]
cct_gens = parse_gens(cct_ibr_generators)
if cct_gens:
    args += ["--ibr"] + [str(g) for g in cct_gens]
out = subprocess.run(args, capture_output=True, text=True)
print(out.stdout)
if out.returncode != 0:
    print("The command failed:")
    print(out.stderr[-2000:])
png = f"cct_output/cct_b{cct_bus}_equal_area.png"
if os.path.exists(png):
    display(Image(filename=png))

## Next steps

- Explore every combination without running anything: [Build your own grid](https://ishaibu.github.io/ibr-faultscreen/demo/#explore),
  then test yourself with the [quiz](https://ishaibu.github.io/ibr-faultscreen/demo/#quiz).
- Install on your own computer to screen thousands of scenarios and train the screening
  model: `pip install ibr-faultscreen`, then `ibr-faultscreen --help`.
- Read how it works and its limitations in the
  [README](https://github.com/ishaibu/ibr-faultscreen#readme) and the
  [design specification](https://github.com/ishaibu/ibr-faultscreen/blob/main/docs/design-spec.md).
- Questions or ideas: [open an issue](https://github.com/ishaibu/ibr-faultscreen/issues).
  Please never upload confidential network models.

If you use IBR-FaultScreen, please cite it: [doi.org/10.5281/zenodo.23290219](https://doi.org/10.5281/zenodo.23290219).